# AIML-01 Fraud Detection: Day 5-6, XGBoost + MLflow

**Goal (portal step 3):** train an XGBoost model with **class weights**, evaluate with **precision-recall AUC (not accuracy)**, and **track every run in MLflow**.

**Three decisions that make this honest (be ready to explain each):**
1. **Time-based split.** Train on the earliest hours, validate on the middle, test on the latest. A real fraud system only ever sees the past, so a random split would flatter the model.
2. **Natural class ratio.** Validation and test keep the real ~0.3% fraud rate. We do not shrink the normal class when testing.
3. **With and without the balance columns.** Your Day 1-2 check showed they are very strong and may leak. We train both ways and report both.

Run cells top to bottom.

## 1. Setup

In [ ]:
!pip install -q mlflow xgboost scikit-learn pyarrow
import os, json, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
import mlflow
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve

os.makedirs("outputs", exist_ok=True); os.makedirs("figures", exist_ok=True)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
plt.rcParams.update({"figure.figsize": (7, 4), "axes.spines.top": False, "axes.spines.right": False})

## 2. Load the Day 3-4 features
If you are still in the same Colab session as Day 3-4, the file is already there. If not, upload `day3-4_outputs.zip` when asked.

In [ ]:
FEATURES_PATH = "outputs/features_full.parquet"
if not os.path.exists(FEATURES_PATH):
    from google.colab import files
    up = files.upload()                      # choose day3-4_outputs.zip
    !unzip -oq day3-4_outputs.zip
feats = pd.read_parquet(FEATURES_PATH)
groups = json.load(open("outputs/feature_groups.json"))
BASE, BALANCE = groups["base"], groups["balance"]
print(f"{len(feats):,} rows | {int(feats.isFraud.sum()):,} fraud ({100*feats.isFraud.mean():.3f}%)")
print("Base features   :", BASE)
print("Balance features:", BALANCE)

## 3. Time-based split

In [ ]:
max_step = int(feats["step"].max())
t1, t2 = int(max_step * 0.70), int(max_step * 0.85)
train = feats[feats.step <= t1]
valid = feats[(feats.step > t1) & (feats.step <= t2)]
test  = feats[feats.step > t2]

for name, d in [("train", train), ("valid", valid), ("test", test)]:
    print(f"{name:5s} steps {int(d.step.min()):>3}-{int(d.step.max()):>3} | {len(d):>9,} rows | {int(d.isFraud.sum()):>5,} fraud | {100*d.isFraud.mean():.3f}%")
print(f"\nRandom guessing would score a PR-AUC of about {test.isFraud.mean():.4f} on the test set.")

## 4. The experiment function
Each call trains one model and logs it to MLflow: the settings (parameters), the scores (metrics), and the model file.

**Metrics:**
- **PR-AUC** (`average_precision_score`): the main metric. Accuracy would be 99.7% for a model that never flags anything.
- **ROC-AUC**: reported for reference.
- **Threshold chosen on validation** (best F1), then applied unchanged to the test set. Never pick a threshold on the test set.
- **Recall at 1% alerts**: if analysts can only review the 1% highest-scored transactions, what share of fraud do we catch?

In [ ]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("fraud-detection-paysim")

def best_f1_threshold(y, p):
    prec, rec, thr = precision_recall_curve(y, p)
    f1 = 2 * prec[:-1] * rec[:-1] / np.clip(prec[:-1] + rec[:-1], 1e-12, None)
    i = int(np.nanargmax(f1))
    return float(thr[i]), float(f1[i])

def recall_at_alert_rate(y, p, rate=0.01):
    k = max(1, int(len(y) * rate))
    top = np.argsort(-p)[:k]
    return float(y.to_numpy()[top].sum() / max(1, y.sum()))

def run_experiment(run_name, feature_cols, weighted):
    Xtr, ytr = train[feature_cols], train["isFraud"]
    Xva, yva = valid[feature_cols], valid["isFraud"]
    Xte, yte = test[feature_cols], test["isFraud"]
    spw = float((ytr == 0).sum() / max(1, (ytr == 1).sum())) if weighted else 1.0

    params = dict(n_estimators=400, max_depth=6, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8,
                  tree_method="hist", eval_metric="aucpr", early_stopping_rounds=30,
                  scale_pos_weight=spw, random_state=42, n_jobs=-1)
    with mlflow.start_run(run_name=run_name):
        model = xgb.XGBClassifier(**params)
        model.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
        pva, pte = model.predict_proba(Xva)[:, 1], model.predict_proba(Xte)[:, 1]

        thr, f1_val = best_f1_threshold(yva, pva)
        flag = pte >= thr
        tp = int((flag & (yte.to_numpy() == 1)).sum())
        precision = tp / max(1, int(flag.sum())); recall = tp / max(1, int(yte.sum()))

        metrics = {
            "val_pr_auc": average_precision_score(yva, pva),
            "test_pr_auc": average_precision_score(yte, pte),
            "test_roc_auc": roc_auc_score(yte, pte),
            "threshold_from_val": thr,
            "test_precision": precision,
            "test_recall": recall,
            "test_recall_at_1pct_alerts": recall_at_alert_rate(yte, pte, 0.01),
            "best_iteration": int(model.best_iteration),
        }
        mlflow.log_params({"features": run_name.split("|")[0].strip(), "n_features": len(feature_cols),
                           "class_weighting": weighted, "scale_pos_weight": round(spw, 2),
                           "max_depth": params["max_depth"], "learning_rate": params["learning_rate"],
                           "split": f"time: train<= {t1}, valid<= {t2}, test> {t2}"})
        mlflow.log_metrics({k: float(v) for k, v in metrics.items()})
        model.save_model("model_tmp.json"); mlflow.log_artifact("model_tmp.json")
    return dict(run=run_name, model=model, features=feature_cols, weighted=weighted, thr=thr,
                p_test=pte, **metrics)

## 5. Run the four experiments
`base` = no balance columns. `base + balance` = everything. Each is run **with** and **without** class weights (the weight tells XGBoost that one fraud matters as much as roughly 340 normal transactions).

In [ ]:
runs = []
for fname, cols in [("base", BASE), ("base+balance", BASE + BALANCE)]:
    for weighted in (False, True):
        name = f"{fname} | {'weighted' if weighted else 'unweighted'}"
        r = run_experiment(name, cols, weighted)
        runs.append(r)
        print(f"{name:28s} val PR-AUC {r['val_pr_auc']:.4f} | test PR-AUC {r['test_pr_auc']:.4f} | recall@1% {r['test_recall_at_1pct_alerts']:.1%}")

## 6. Results table (this goes in your README)

In [ ]:
res = pd.DataFrame([{k: v for k, v in r.items() if k not in ("model", "features", "p_test", "weighted")} for r in runs])
res = res.set_index("run")
cols_show = ["val_pr_auc", "test_pr_auc", "test_roc_auc", "test_precision", "test_recall", "test_recall_at_1pct_alerts"]
display(res[cols_show])
res.to_csv("outputs/results.csv")
print(f"Test-set baseline PR-AUC (random guessing): {test.isFraud.mean():.4f}")

**How to read this (write your own sentences in your notes):**
- Compare `base` with `base + balance`. A big jump when the balance columns are added means the model leans on them. Report both honestly.
- Compare weighted with unweighted. Class weights usually raise recall and lower precision. Which one matters more to a fraud analyst?
- PR-AUC far above the random baseline means the model has real signal.

## 7. Precision-recall curves on the test set

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for r in runs:
    p, rc, _ = precision_recall_curve(test["isFraud"], r["p_test"])
    ax.plot(rc, p, label=f"{r['run']} (AP {r['test_pr_auc']:.3f})")
ax.axhline(test.isFraud.mean(), color="grey", ls="--", lw=1, label="random guessing")
ax.set_xlabel("Recall (share of fraud caught)"); ax.set_ylabel("Precision (share of alerts that are fraud)")
ax.set_title("Precision-recall on the test set"); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig("figures/05_pr_curves.png", dpi=150); plt.show()

## 8. Choose the final model
Rule: pick by **validation** PR-AUC, never by the test score. Among the runs that do **not** use the balance columns, we pick the best one, so the live system does not depend on possibly leaky fields. If you prefer the best overall, change `allow_balance` to `True` and explain why in your README.

In [ ]:
allow_balance = False
cands = [r for r in runs if allow_balance or r["features"] == BASE]
best = max(cands, key=lambda r: r["val_pr_auc"])
print("Chosen:", best["run"], "| val PR-AUC", round(best["val_pr_auc"], 4), "| test PR-AUC", round(best["test_pr_auc"], 4))

best["model"].save_model("outputs/model.json")
card = {"run": best["run"], "features": best["features"], "threshold": best["thr"],
        "class_weighting": best["weighted"],
        "test_pr_auc": best["test_pr_auc"], "test_precision": best["test_precision"],
        "test_recall": best["test_recall"], "test_recall_at_1pct_alerts": best["test_recall_at_1pct_alerts"],
        "split": {"train_max_step": t1, "valid_max_step": t2}}
json.dump(card, open("outputs/model_card.json", "w"), indent=2)
print(json.dumps(card, indent=2))

## 9. Feature importance of the chosen model

In [ ]:
imp = pd.Series(best["model"].get_booster().get_score(importance_type="gain")).sort_values()
fig, ax = plt.subplots()
imp.plot.barh(ax=ax, color="#4C78A8"); ax.set_xlabel("Gain"); ax.set_title("What the chosen model relies on")
fig.tight_layout(); fig.savefig("figures/06_feature_importance.png", dpi=150); plt.show()

## 10. What MLflow recorded

In [ ]:
runs_df = mlflow.search_runs(experiment_names=["fraud-detection-paysim"])
show = [c for c in runs_df.columns if c.startswith(("tags.mlflow.runName", "params.features", "params.class_weighting", "metrics.test_pr_auc", "metrics.val_pr_auc"))]
display(runs_df[show])
print(f"{len(runs_df)} runs are stored in mlflow.db (this is the 'track all runs in MLflow' deliverable).")

## 11. Download your files

In [ ]:
!zip -qr day5-6_outputs.zip outputs figures mlflow.db mlruns 2>/dev/null
try:
    from google.colab import files
    files.download("day5-6_outputs.zip")
except ImportError:
    print("Not in Colab: the zip is saved next to the notebook.")

## Day 5-6 checklist
- [ ] All four runs finished and the results table is shown
- [ ] You can explain why PR-AUC, not accuracy
- [ ] You can explain the time-based split and why validation picked the threshold
- [ ] You wrote one honest sentence about `base` vs `base + balance`
- [ ] `day5-6_outputs.zip` downloaded (contains `model.json` for the API)

**Next (Day 7-8):** a FastAPI service that loads `model.json`, reads receiver history from Feast, and returns a fraud score, plus a Kafka producer that replays transactions into it.